In [ ]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from tqdm import tqdm

import rasterio
from rasterio.features import rasterize
from rasterio.windows import from_bounds

from shapely import wkt
from shapely.geometry import box
from shapely.geometry import mapping

os.chdir('/store/carroll/sbgplants/')

In [24]:
# file paths
raw = 'data/raw'
rdn_fol = '/store/carroll/col/data/2018/raw/L1/'
out_folder = 'data/out_csv'

table = 'pixel'

In [41]:
# load relevant data
raster_plot_event = gpd.read_file(os.path.join(out_folder, 'raster_plot_event.geojson'))
fids = raster_plot_event.granule_id.unique()

In [43]:
out = []

for fid in tqdm(fids):
    tmp = raster_plot_event[raster_plot_event['granule_id']==fid] # filter raster_plot_event to a single fid

    # rasterize filtered gdf
    fp = glob(os.path.join(rdn_fol, f'*/{fid}_rdn_ort_igm_ort'))[0]
    shapes = [(mapping(geom), val) for geom, val in zip(tmp.geometry, tmp.index)]
    with rasterio.open(fp) as src:
        r = rasterize(shapes, out_shape=(src.height, src.width), transform=src.transform, all_touched=False, nodata=-9999, fill=-9999)
        nodata = src.read(1)<=0 # nodata mask
    r[nodata] = -9999
    
    # extract glt_row, glt_col, plot_name
    mask = r!=-9999
    row, col = np.where(mask)
    val = r[mask] # raster plot ids
    
    df = pd.DataFrame({
        'raster_plot_id': val,
        'granule_id': fid,
        'glt_row': row,
        'glt_column': col
    })
    out.append(df)

out_table = pd.concat(out)
out_table['pixel_id'] = range(len(out_table))
out_table = out_table[['pixel_id', 'raster_plot_id', 'granule_id', 'glt_row', 'glt_column']]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 62/62 [13:08<00:00, 12.71s/it]


In [48]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)